# 02 - Silver: RUES (Registro Mercantil)

Trabajo Práctico 2 - Transformación Bronze → Silver con reglas de calidad (DQL)

## Objetivo

Tipar y validar `Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api` (2.815.313 filas). Las filas que pasan todas las reglas de calidad van a `Datos_Empresas.silver.empresas_rues`; las que incumplen al menos una van a `Datos_Empresas.dql.rechazados_rues` con el motivo del rechazo.

## Transformaciones de tipado

* `fecha_matricula`, `fecha_renovacion`, `fecha_cancelacion`: texto `YYYYMMDD` → `DATE` (si no tienen 8 dígitos o no son una fecha de calendario válida, quedan `NULL`).
* `fecha_vigencia`: igual, pero además el valor centinela `99991231` ("sin vencimiento") se convierte explícitamente en `NULL`.
* `fecha_actualizacion`: texto `YYYY/MM/DD HH:MM:SS.fffffffff` → `TIMESTAMP`.
* `ultimo_ano_renovado`: texto → `INT`.
* `matricula` y `numero_identificacion`: se recortan espacios (`trim`) antes de validar.
* El resto de columnas se conserva como texto (son identificadores o categorías, no tiene sentido tipar como número).

## Reglas de calidad (rechazo)

| # | Regla | Justificación |
|---|---|---|
| 1 | `matricula` o `numero_identificacion` vacíos | Son las columnas que identifican a la empresa; sin ellas el registro no se puede usar ni enlazar en Gold. |
| 2 | Combinación (`codigo_camara`, `matricula`) duplicada | Regla de unicidad: una empresa no puede tener más de un registro activo con la misma matrícula en la misma cámara. |
| 3 | `fecha_matricula` con formato/calendario inválido o en el futuro respecto a hoy | Una fecha de matrícula mal formada o futura es imposible en la realidad del negocio — ya se detectaba como outlier en el TP1 (notebook de validaciones); aquí se convierte en regla de rechazo activa. |
| 4 | `cod_ciiu_act_econ_pri` vacío | Sin código de actividad económica principal no se puede clasificar la empresa por sector, que es justo lo que necesitamos para la pregunta de negocio "actividades económicas más frecuentes por región" en Gold. Se verificó en vivo contra Bronze: afecta solo 31 de 2.815.313 filas (~0.001%), y el 100% de esas 31 son empresas con `estado_matricula = CANCELADA` o `MATRÍCULA CANCELADA LEY 1429` — el dato ya llega vacío desde la fuente para registros cancelados, no es un error de nuestra transformación. |

Un registro puede incumplir varias reglas a la vez; todas quedan registradas en `_motivo_rechazo`, separadas por `; `.

In [ ]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TABLA_BRONZE = "Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api"
TABLA_SILVER = "Datos_Empresas.silver.empresas_rues"
TABLA_DQL = "Datos_Empresas.dql.rechazados_rues"

df_bronze = spark.table(TABLA_BRONZE)
print(f"Bronze: {df_bronze.count():,} filas")

---

## Paso 1: Tipado seguro de columnas

In [ ]:
%python
FORMATO_FECHA_ACTUALIZACION = "yyyy/MM/dd HH:mm:ss.SSSSSSSSS"

# IMPORTANTE: se usa try_to_date / try_to_timestamp (no to_date / to_timestamp).
# Con ANSI SQL activado (el modo por defecto en los SQL Warehouse de Databricks),
# to_date lanza una excepción ante una fecha invalida como '99999997' (mes 99),
# en vez de devolver NULL. try_to_date siempre devuelve NULL si no puede parsear,
# tanto si el string no tiene 8 dígitos como si los 8 dígitos no forman una fecha
# de calendario válida (eso se verificó probando esta consulta en vivo: 533 filas
# de RUES tienen 8 dígitos pero un mes/día imposible).
df_tipado = (
    df_bronze
    .withColumn("matricula", F.trim(F.col("matricula")))
    .withColumn("numero_identificacion", F.trim(F.col("numero_identificacion")))
    .withColumn("fecha_matricula", F.expr("try_to_date(fecha_matricula, 'yyyyMMdd')"))
    .withColumn("fecha_renovacion", F.expr("try_to_date(fecha_renovacion, 'yyyyMMdd')"))
    .withColumn(
        "fecha_vigencia",
        F.when(F.col("fecha_vigencia") == "99991231", F.lit(None))
         .otherwise(F.expr("try_to_date(fecha_vigencia, 'yyyyMMdd')")),
    )
    .withColumn("fecha_cancelacion", F.expr("try_to_date(fecha_cancelacion, 'yyyyMMdd')"))
    .withColumn("ultimo_ano_renovado", F.col("ultimo_ano_renovado").cast("int"))
    .withColumn(
        "fecha_actualizacion",
        F.expr(f"try_to_timestamp(fecha_actualizacion, '{FORMATO_FECHA_ACTUALIZACION}')"),
    )
)

df_tipado.select("matricula", "fecha_matricula", "fecha_vigencia", "fecha_actualizacion").show(5, truncate=False)

---

## Paso 1.5: Diagnóstico previo — evidencia detrás de cada regla

Antes de aplicar las reglas de calidad, se corre el query que justifica cada una contra Bronze. Esto es lo que se corrió para decidir las 4 reglas del notebook (puedes volver a ejecutarlas para verlo tú mismo).

In [ ]:
%sql
-- Evidencia Regla 1: campos obligatorios vacíos
SELECT COUNT(*) AS filas_afectadas
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api
WHERE TRIM(matricula) IS NULL OR TRIM(matricula) = ''
   OR TRIM(numero_identificacion) IS NULL OR TRIM(numero_identificacion) = '';

In [ ]:
%sql
-- Evidencia Regla 2: (codigo_camara, matricula) duplicada
SELECT codigo_camara, matricula, COUNT(*) AS repeticiones
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api
GROUP BY codigo_camara, matricula
HAVING COUNT(*) > 1
ORDER BY repeticiones DESC
LIMIT 10;

In [ ]:
%sql
-- Evidencia Regla 3: fecha_matricula con formato/calendario inválido o en el futuro
SELECT
  SUM(CASE WHEN try_to_date(fecha_matricula, 'yyyyMMdd') IS NULL THEN 1 ELSE 0 END) AS formato_o_calendario_invalido,
  SUM(CASE WHEN try_to_date(fecha_matricula, 'yyyyMMdd') > current_date() THEN 1 ELSE 0 END) AS fecha_futura
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api;

In [ ]:
%sql
-- Evidencia Regla 4: cod_ciiu_act_econ_pri vacío -> se agrupa por estado_matricula
-- para confirmar que son empresas canceladas (hallazgo: 31 filas, 100% CANCELADA)
SELECT estado_matricula, COUNT(*) AS cantidad
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api
WHERE cod_ciiu_act_econ_pri IS NULL OR TRIM(cod_ciiu_act_econ_pri) = ''
GROUP BY estado_matricula
ORDER BY cantidad DESC;

---

## Paso 2: Aplicar las reglas de calidad y calcular `_motivo_rechazo`

In [ ]:
%python
hoy = F.current_date()

w_matricula = Window.partitionBy("codigo_camara", "matricula")

df_reglas = (
    df_tipado
    .withColumn("_dup_count", F.count("*").over(w_matricula))
    # Regla 1: campos obligatorios vacíos
    .withColumn(
        "_r1",
        F.when(
            (F.col("matricula").isNull()) | (F.col("matricula") == "")
            | (F.col("numero_identificacion").isNull()) | (F.col("numero_identificacion") == ""),
            F.lit("campo_obligatorio_vacio"),
        ),
    )
    # Regla 2: (codigo_camara, matricula) duplicada
    .withColumn("_r2", F.when(F.col("_dup_count") > 1, F.lit("matricula_duplicada")))
    # Regla 3: fecha_matricula con formato/calendario inválido (quedó NULL tras el try_to_date
    # del Paso 1) o en el futuro respecto a hoy
    .withColumn(
        "_r3",
        F.when(F.col("fecha_matricula").isNull(), F.lit("fecha_matricula_formato_invalido"))
         .when(F.col("fecha_matricula") > hoy, F.lit("fecha_matricula_futura")),
    )
    # Regla 4: cod_ciiu_act_econ_pri vacío (verificado contra Bronze: 31 filas, 100% CANCELADA)
    .withColumn(
        "_r4",
        F.when(
            (F.col("cod_ciiu_act_econ_pri").isNull()) | (F.trim(F.col("cod_ciiu_act_econ_pri")) == ""),
            F.lit("cod_ciiu_vacio"),
        ),
    )
    .withColumn("_motivo_rechazo", F.concat_ws("; ", F.col("_r1"), F.col("_r2"), F.col("_r3"), F.col("_r4")))
    .withColumn(
        "_motivo_rechazo",
        F.when(F.col("_motivo_rechazo") == "", F.lit(None)).otherwise(F.col("_motivo_rechazo")),
    )
)

---

## Paso 3: Separar aceptados (Silver) de rechazados (DQL)

In [ ]:
%python
columnas_auxiliares = ["_dup_count", "_r1", "_r2", "_r3", "_r4"]

df_silver = (
    df_reglas.filter(F.col("_motivo_rechazo").isNull())
    .drop(*columnas_auxiliares, "_motivo_rechazo", "_ingested_at", "_source")
    .withColumn("_processed_at", F.current_timestamp())
)

df_dql = (
    df_reglas.filter(F.col("_motivo_rechazo").isNotNull())
    .drop(*columnas_auxiliares)
    .withColumn("_fecha_rechazo", F.current_timestamp())
)

total = df_bronze.count()
aceptados = df_silver.count()
rechazados = df_dql.count()
print(f"Bronze: {total:,} | Silver (aceptados): {aceptados:,} | DQL (rechazados): {rechazados:,}")
assert total == aceptados + rechazados, "Se perdieron o duplicaron filas en la separación Silver/DQL"

---

## Paso 4: Persistir (idempotente: `overwrite` recalcula la tabla completa en cada corrida)

In [ ]:
%python
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_SILVER)
df_dql.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_DQL)

print(f"Escrito: {TABLA_SILVER}")
print(f"Escrito: {TABLA_DQL}")

---

## Paso 5: Validación

In [ ]:
%sql
SELECT COUNT(*) AS total_silver FROM Datos_Empresas.silver.empresas_rues;

SELECT COUNT(*) AS total_dql FROM Datos_Empresas.dql.rechazados_rues;

In [ ]:
%sql
-- Distribución de rechazos por motivo (un registro puede aparecer en más de un motivo compuesto)
SELECT _motivo_rechazo, COUNT(*) AS cantidad
FROM Datos_Empresas.dql.rechazados_rues
GROUP BY _motivo_rechazo
ORDER BY cantidad DESC
LIMIT 20;

In [ ]:
%sql
DESCRIBE EXTENDED Datos_Empresas.silver.empresas_rues;

In [ ]:
%sql
SELECT * FROM Datos_Empresas.silver.empresas_rues LIMIT 10;

---

## Siguiente paso

Continuar con `03_Silver_SECOP.ipynb`, `04_Silver_CIIU.ipynb` y `05_Silver_TRM.ipynb` (son independientes entre sí), y luego `06_Gold_Empresas_Contratos.ipynb`.